<a href="https://colab.research.google.com/github/harshitkatragadda25/markdown-to-gdocs-converter/blob/main/markdown_to_gdocs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:

"""
This notebook converts markdown meeting notes into a well-formatted Google Doc
with proper styling, checkboxes, and hierarchical structure.

Author: Pavan
Time: ~30 minutes
"""

# Step 1: Install required dependencies
print("📦 Installing dependencies...")
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib -q
print("✅ Dependencies installed!\n")

# Step 2: Import libraries
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import re

# Step 3: Authenticate with Google
print("🔐 Authenticating with Google...")
auth.authenticate_user()
print("✅ Authentication successful!\n")

# Step 4: Define the markdown content
MARKDOWN_CONTENT = """# Product Team Sync - May 15, 2023
## Attendees
- Sarah Chen (Product Lead)
- Mike Johnson (Engineering)
- Anna Smith (Design)
- David Park (QA)
## Agenda
### 1. Sprint Review
* Completed Features
  * User authentication flow
  * Dashboard redesign
  * Performance optimization
    * Reduced load time by 40%
    * Implemented caching solution
* Pending Items
  * Mobile responsive fixes
  * Beta testing feedback integration
### 2. Current Challenges
* Resource constraints in QA team
* Third-party API integration delays
* User feedback on new UI
  * Navigation confusion
  * Color contrast issues
### 3. Next Sprint Planning
* Priority Features
  * Payment gateway integration
  * User profile enhancement
  * Analytics dashboard
* Technical Debt
  * Code refactoring
  * Documentation updates
## Action Items
- [ ] @sarah: Finalize Q3 roadmap by Friday
- [ ] @mike: Schedule technical review for payment integration
- [ ] @anna: Share updated design system documentation
- [ ] @david: Prepare QA resource allocation proposal
## Next Steps
* Schedule individual team reviews
* Update sprint board
* Share meeting summary with stakeholders
## Notes
* Next sync scheduled for May 22, 2023
* Platform demo for stakeholders on May 25
* Remember to update JIRA tickets
---
Meeting recorded by: Sarah Chen
Duration: 45 minutes
"""

# Step 5: Parse markdown into structured elements
class MarkdownParser:
    """Parse markdown content into structured elements for Google Docs"""

    def __init__(self, content):
        self.content = content
        self.elements = []

    def parse(self):
        """Parse markdown and return structured elements"""
        lines = self.content.split('\n')
        i = 0

        while i < len(lines):
            line = lines[i]


            if not line.strip():
                i += 1
                continue


            if line.strip() == '---':
                self.elements.append({'type': 'separator'})
                i += 1
                continue


            if line.startswith('#'):
                level = len(re.match(r'^#+', line).group())
                text = line.lstrip('#').strip()
                self.elements.append({
                    'type': f'heading{level}',
                    'text': text
                })
                i += 1
                continue


            checkbox_match = re.match(r'^- \[ \] (.+)$', line)
            if checkbox_match:
                text = checkbox_match.group(1)

                mention_match = re.match(r'^@(\w+): (.+)$', text)
                if mention_match:
                    assignee = mention_match.group(1)
                    task = mention_match.group(2)
                    self.elements.append({
                        'type': 'checkbox',
                        'text': task,
                        'assignee': assignee
                    })
                else:
                    self.elements.append({
                        'type': 'checkbox',
                        'text': text
                    })
                i += 1
                continue


            bullet_match = re.match(r'^(\s*)([*-]) (.+)$', line)
            if bullet_match:
                indent = len(bullet_match.group(1))
                text = bullet_match.group(3)
                nesting_level = indent // 2
                self.elements.append({
                    'type': 'bullet',
                    'text': text,
                    'level': nesting_level
                })
                i += 1
                continue


            self.elements.append({
                'type': 'text',
                'text': line.strip()
            })
            i += 1

        return self.elements

# Step 6: Create Google Doc with formatting
class GoogleDocsFormatter:
    """Format and create Google Docs with proper styling"""

    def __init__(self):
        self.service = build('docs', 'v1')
        self.requests = []
        self.current_index = 1

    def create_document(self, title):
        """Create a new Google Doc"""
        try:
            doc = self.service.documents().create(body={'title': title}).execute()
            self.doc_id = doc.get('documentId')
            print(f"✅ Created Google Doc: {title}")
            print(f"📄 Document ID: {self.doc_id}")
            print(f"🔗 URL: https://docs.google.com/document/d/{self.doc_id}/edit\n")
            return self.doc_id
        except HttpError as error:
            print(f"❌ Error creating document: {error}")
            raise

    def add_element(self, element):
        """Add an element to the document"""

        if element['type'] == 'separator':

            self.requests.append({
                'insertText': {
                    'location': {'index': self.current_index},
                    'text': '\n'
                }
            })
            self.current_index += 1
            return

        if element['type'] == 'heading1':
            text = element['text'] + '\n'
            self.requests.append({
                'insertText': {
                    'location': {'index': self.current_index},
                    'text': text
                }
            })

            self.requests.append({
                'updateParagraphStyle': {
                    'range': {
                        'startIndex': self.current_index,
                        'endIndex': self.current_index + len(text) - 1
                    },
                    'paragraphStyle': {
                        'namedStyleType': 'HEADING_1'
                    },
                    'fields': 'namedStyleType'
                }
            })
            self.current_index += len(text)

        elif element['type'] == 'heading2':
            text = element['text'] + '\n'
            self.requests.append({
                'insertText': {
                    'location': {'index': self.current_index},
                    'text': text
                }
            })

            self.requests.append({
                'updateParagraphStyle': {
                    'range': {
                        'startIndex': self.current_index,
                        'endIndex': self.current_index + len(text) - 1
                    },
                    'paragraphStyle': {
                        'namedStyleType': 'HEADING_2'
                    },
                    'fields': 'namedStyleType'
                }
            })
            self.current_index += len(text)

        elif element['type'] == 'heading3':
            text = element['text'] + '\n'
            self.requests.append({
                'insertText': {
                    'location': {'index': self.current_index},
                    'text': text
                }
            })

            self.requests.append({
                'updateParagraphStyle': {
                    'range': {
                        'startIndex': self.current_index,
                        'endIndex': self.current_index + len(text) - 1
                    },
                    'paragraphStyle': {
                        'namedStyleType': 'HEADING_3'
                    },
                    'fields': 'namedStyleType'
                }
            })
            self.current_index += len(text)

        elif element['type'] == 'checkbox':

            if 'assignee' in element:
                assignee_text = f"@{element['assignee']}: "
                task_text = element['text'] + '\n'
                full_text = assignee_text + task_text


                self.requests.append({
                    'insertText': {
                        'location': {'index': self.current_index},
                        'text': full_text
                    }
                })


                self.requests.append({
                    'updateTextStyle': {
                        'range': {
                            'startIndex': self.current_index,
                            'endIndex': self.current_index + len(assignee_text)
                        },
                        'textStyle': {
                            'bold': True,
                            'foregroundColor': {
                                'color': {
                                    'rgbColor': {
                                        'red': 0.0,
                                        'green': 0.4,
                                        'blue': 0.8
                                    }
                                }
                            }
                        },
                        'fields': 'bold,foregroundColor'
                    }
                })


                self.requests.append({
                    'createParagraphBullets': {
                        'range': {
                            'startIndex': self.current_index,
                            'endIndex': self.current_index + len(full_text)
                        },
                        'bulletPreset': 'BULLET_CHECKBOX'
                    }
                })

                self.current_index += len(full_text)
            else:
                text = element['text'] + '\n'
                self.requests.append({
                    'insertText': {
                        'location': {'index': self.current_index},
                        'text': text
                    }
                })

                self.requests.append({
                    'createParagraphBullets': {
                        'range': {
                            'startIndex': self.current_index,
                            'endIndex': self.current_index + len(text)
                        },
                        'bulletPreset': 'BULLET_CHECKBOX'
                    }
                })
                self.current_index += len(text)

        elif element['type'] == 'bullet':
            text = element['text'] + '\n'
            self.requests.append({
                'insertText': {
                    'location': {'index': self.current_index},
                    'text': text
                }
            })

            self.requests.append({
                'createParagraphBullets': {
                    'range': {
                        'startIndex': self.current_index,
                        'endIndex': self.current_index + len(text)
                    },
                    'bulletPreset': 'BULLET_DISC_CIRCLE_SQUARE'
                }
            })

            if element['level'] > 0:
                self.requests.append({
                    'updateParagraphStyle': {
                        'range': {
                            'startIndex': self.current_index,
                            'endIndex': self.current_index + len(text)
                        },
                        'paragraphStyle': {
                            'indentStart': {
                                'magnitude': 36 * element['level'],
                                'unit': 'PT'
                            }
                        },
                        'fields': 'indentStart'
                    }
                })
            self.current_index += len(text)

        elif element['type'] == 'text':
            text = element['text'] + '\n'
            self.requests.append({
                'insertText': {
                    'location': {'index': self.current_index},
                    'text': text
                }
            })

            self.requests.append({
                'updateTextStyle': {
                    'range': {
                        'startIndex': self.current_index,
                        'endIndex': self.current_index + len(text) - 1
                    },
                    'textStyle': {
                        'italic': True,
                        'foregroundColor': {
                            'color': {
                                'rgbColor': {
                                    'red': 0.5,
                                    'green': 0.5,
                                    'blue': 0.5
                                }
                            }
                        }
                    },
                    'fields': 'italic,foregroundColor'
                }
            })
            self.current_index += len(text)

    def apply_formatting(self):
        """Apply all formatting requests to the document"""
        try:
            if self.requests:
                self.service.documents().batchUpdate(
                    documentId=self.doc_id,
                    body={'requests': self.requests}
                ).execute()
                print(f"✅ Applied {len(self.requests)} formatting operations")
        except HttpError as error:
            print(f"❌ Error applying formatting: {error}")
            raise

# Step 7: Main execution
def main():
    """Main function to convert markdown to Google Doc"""
    try:
        print("\n🚀 Starting Markdown to Google Docs Conversion\n")
        print("=" * 60)


        print("\n📝 Parsing markdown content...")
        parser = MarkdownParser(MARKDOWN_CONTENT)
        elements = parser.parse()
        print(f"✅ Parsed {len(elements)} elements\n")


        print("📄 Creating Google Doc...")
        formatter = GoogleDocsFormatter()
        doc_id = formatter.create_document("Product Team Sync - May 15, 2023")

        print("🎨 Applying formatting...")
        for element in elements:
            formatter.add_element(element)

        formatter.apply_formatting()

        print("\n" + "=" * 60)
        print("✅ Conversion completed successfully!")
        print("=" * 60)
        print(f"\n🔗 Open your document here:")
        print(f"   https://docs.google.com/document/d/{doc_id}/edit\n")

        return doc_id

    except Exception as e:
        print(f"\n❌ Error during conversion: {str(e)}")
        import traceback
        traceback.print_exc()
        raise

# Run the converter
if __name__ == "__main__":
    doc_id = main()

📦 Installing dependencies...
✅ Dependencies installed!

🔐 Authenticating with Google...
✅ Authentication successful!


🚀 Starting Markdown to Google Docs Conversion


📝 Parsing markdown content...
✅ Parsed 47 elements

📄 Creating Google Doc...


✅ Created Google Doc: Product Team Sync - May 15, 2023
📄 Document ID: 1sNaJ0RQLD9J-t1pMjCgz-xRhpItm7YeU85RxMUNaStw
🔗 URL: https://docs.google.com/document/d/1sNaJ0RQLD9J-t1pMjCgz-xRhpItm7YeU85RxMUNaStw/edit

🎨 Applying formatting...
✅ Applied 111 formatting operations

✅ Conversion completed successfully!

🔗 Open your document here:
   https://docs.google.com/document/d/1sNaJ0RQLD9J-t1pMjCgz-xRhpItm7YeU85RxMUNaStw/edit

